In [24]:
import pandas as pd
import numpy as np

In [25]:
gdsc = pd.read_csv("datasets/GDSC_DATASET.csv")
gdsc.head()

,COSMIC_ID,CELL_LINE_NAME,TCGA_DESC,DRUG_ID,DRUG_NAME,LN_IC50,AUC,Z_SCORE,GDSC Tissue descriptor 1,GDSC Tissue descriptor 2,Cancer Type (matching TCGA label),Microsatellite instability Status (MSI),Screen Medium,Growth Properties,CNA,Gene Expression,Methylation,TARGET,TARGET_PATHWAY
0,683667,PFSK-1,MB,1003,Camptothecin,-1.463887,0.930220,0.433123,nervous_system,medulloblastoma,MB,MSS/MSI-L,R,Adherent,Y,Y,Y,TOP1,DNA replication
1,684057,ES5,UNCLASSIFIED,1003,Camptothecin,-3.360586,0.791072,-0.599569,bone,ewings_sarcoma,NaN,MSS/MSI-L,R,Adherent,Y,Y,Y,TOP1,DNA replication
2,684059,ES7,UNCLASSIFIED,1003,Camptothecin,-5.044940,0.592660,-1.516647,bone,ewings_sarcoma,NaN,MSS/MSI-L,R,Adherent,Y,Y,Y,TOP1,DNA replication
3,684062,EW-11,UNCLASSIFIED,1003,Camptothecin,-3.741991,0.734047,-0.807232,bone,ewings_sarcoma,NaN,MSS/MSI-L,R,Adherent,Y,Y,Y,TOP1,DNA replication
4,684072,SK-ES-1,UNCLASSIFIED,1003,Camptothecin,-5.142961,0.582439,-1.570016,bone,ewings_sarcoma,NaN,MSS/MSI-L,R,Semi-Adherent,Y,Y,Y,TOP1,DNA replication


In [26]:
for col in gdsc.columns:
    if 'IC' in col.upper() or 'AUC' in col.upper():
        print(col)


COSMIC_ID
LN_IC50
AUC
Microsatellite instability Status (MSI)


In [27]:
X = gdsc.drop(columns=['DRUG_ID', 'LN_IC50'])
y = gdsc['LN_IC50']


In [28]:
print(X.shape)
print(y.shape)


(242035, 17)
(242035,)


In [29]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [30]:
X.dtypes

COSMIC_ID                                    int64
CELL_LINE_NAME                              object
TCGA_DESC                                   object
DRUG_NAME                                   object
AUC                                        float64
Z_SCORE                                    float64
GDSC Tissue descriptor 1                    object
GDSC Tissue descriptor 2                    object
Cancer Type (matching TCGA label)           object
Microsatellite instability Status (MSI)     object
Screen Medium                               object
Growth Properties                           object
CNA                                         object
Gene Expression                             object
Methylation                                 object
TARGET                                      object
TARGET_PATHWAY                              object
dtype: object

In [31]:
X = X.select_dtypes(include=['float64', 'int64'])

In [32]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [37]:
from sklearn.ensemble import RandomForestRegressor

drug_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

drug_model.fit(X_train, y_train)


,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [38]:
print(X_train.shape)
print(X_train.dtypes.unique())


(193628, 3)
[dtype('int64') dtype('float64')]


In [39]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

y_pred = drug_model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))


MAE: 1.1086309088576032
RMSE: 1.6247345563389173


In [40]:
sample_patient = X.iloc[0:1]


In [41]:
gdsc['Predicted_IC50'] = drug_model.predict(X)


In [43]:
print(gdsc.columns)


Index(['COSMIC_ID', 'CELL_LINE_NAME', 'TCGA_DESC', 'DRUG_ID', 'DRUG_NAME',
       'LN_IC50', 'AUC', 'Z_SCORE', 'GDSC Tissue descriptor 1',
       'GDSC Tissue descriptor 2', 'Cancer Type (matching TCGA label)',
       'Microsatellite instability Status (MSI)', 'Screen Medium',
       'Growth Properties', 'CNA', 'Gene Expression', 'Methylation', 'TARGET',
       'TARGET_PATHWAY', 'Predicted_IC50'],
      dtype='object')


In [44]:
# Find row with lowest predicted IC50
best_index = gdsc['Predicted_IC50'].idxmin()
best_drug_row = gdsc.loc[best_index]

best_drug_id = best_drug_row['DRUG_ID']
best_drug_name = best_drug_row['DRUG_NAME']

print("Best Drug ID:", best_drug_id)
print("Recommended Drug:", best_drug_name)


Best Drug ID: 1079
Recommended Drug: Dasatinib


In [45]:
import joblib

joblib.dump(drug_model, "drug_model.pkl")
print("Drug model saved successfully")


Drug model saved successfully
